In [0]:
%sql
CREATE TABLE practice1.test.emp(
  emp_id      INT PRIMARY KEY,
  name        VARCHAR(50),
  department  VARCHAR(50),
  salary      INT,
  hire_date   DATE,
  manager_id  INT
);

INSERT INTO practice1.test.emp VALUES
(1,'Alice','Engineering',95000,'2019-03-15',NULL),
(2,'Bob','Engineering',82000,'2020-07-01',1),
(3,'Carol','Engineering',91000,'2018-11-20',1),
(4,'David','Sales',70000,'2021-01-10',NULL),
(5,'Eve','Sales',74000,'2020-05-22',4),
(6,'Frank','Sales',68000,'2022-03-08',4),
(7,'Grace','HR',62000,'2021-09-14',NULL),
(8,'Hank','HR',60000,'2023-02-01',7),
(9,'Iris','Engineering',88000,'2021-06-30',1),
(10,'Jake','Sales',73000,'2019-12-05',4);

CREATE TABLE practice1.test.sales (
  sale_id   INT PRIMARY KEY,
  emp_id    INT,
  sale_date DATE,
  amount    INT,
  region    VARCHAR(20)
);

INSERT INTO practice1.test.sales VALUES
(101,4,'2024-01-05',1200,'North'),
(102,5,'2024-01-08',3400,'South'),
(103,6,'2024-01-12',800,'East'),
(104,4,'2024-01-15',2100,'North'),
(105,5,'2024-02-02',1500,'South'),
(106,10,'2024-02-10',4200,'West'),
(107,6,'2024-02-14',950,'East'),
(108,4,'2024-03-01',1800,'North'),
(109,10,'2024-03-05',5100,'West'),
(110,5,'2024-03-20',2900,'South'),
(111,6,'2024-04-01',1100,'East'),
(112,10,'2024-04-15',3700,'West');

In [0]:
%sql
select * from practice1.test.emp;

In [0]:
%sql
select * from practice1.test.sales;

In [0]:
%sql
-- Q1. Assign a unique row number to every employee ordered by salary from highest to lowest.
select row_number() over (order by salary desc) as row_num, emp_id, name, department,salary
from practice1.test.emp;

In [0]:
%sql
--Q2. Within each department, assign a row number to employees ordered by salary descending (1 = highest earner in dept)
select row_number() over (partition by department order by salary desc) as row_num, emp_id, name, department, salary
from practice1.test.emp;


In [0]:
%sql
--Q3. Show each employee's salary and the total salary bill for the entire company alongside every row.
select emp_id, name, department, salary, sum(salary) over () as total_salary
from practice1.test.emp;

In [0]:
%sql
--Q4. For each sale, compute a running total of sale amounts per employee ordered by sale_date
select 
  sale_id,
  emp_id,
  sale_date,
  amount,
  sum(amount) over (
    partition by emp_id 
    order by sale_date
  ) as running_total
from practice1.test.sales;

In [0]:
%sql
--Q5. Show each employee's salary as a percentage of their department's total salary, rounded to 2 decimal places.
select 
  emp_id,
  name,
  department,
  salary,
  round(
    salary * 100.0 / sum(salary) over (partition by department),
    2
  ) as salary_percentage
from practice1.test.emp;

In [0]:
%sql
--Q6. Rank employees by salary within each department
select 
  emp_id,
  name,
  department,
  salary,
  rank() over (partition by department order by salary desc) as rank_in_dept
from practice1.test.emp;

In [0]:
%sql
--Q7. Find employees whose department salary rank is exactly 2 — second highest in their dept.
select *
from (
  select 
    emp_id,
    name,
    department,
    salary,
    rank() over (partition by department order by salary desc) as rank_in_dept
  from practice1.test.emp
) t
where rank_in_dept = 2;

In [0]:
%sql
--Q8. Assign a dense rank to all employees by salary descending
select 
  emp_id,
  name,
  department,
  salary,
  dense_rank() over (order by salary desc) as dense_rank_salary
from practice1.test.emp;

In [0]:
%sql
--Q9. Return only employees who are in the top 2 salary tiers within their department.
select *
from (
  select 
    emp_id,
    name,
    department,
    salary,
    dense_rank() over (partition by department order by salary desc) as dr
  from practice1.test.emp
) t
where dr <= 2;

In [0]:
%sql
--Calculate the difference between the current and previous sale amount per employee.
select 
  sale_id,
  emp_id,
  sale_date,
  amount,
  amount - lag(amount) over (
    partition by emp_id 
    order by sale_date
  ) as diff_from_previous
from practice1.test.sales;

In [0]:
%sql 
--For each employee, show the salary of the person hired just before them in the same department
select emp_id, name, department, salary,
lag(salary) over (partition by department order by hire_date) as prev_emp_salary
from practice1.test.emp;

In [0]:
%sql
--For each employee, show the salary of the next person hired in the same department.
select 
  emp_id,
  name,
  department,
  salary,
  lead(salary) over (
    partition by department 
    order by hire_date
  ) as next_emp_salary
from practice1.test.emp;